<a href="https://colab.research.google.com/github/VikiTarasova/DTA_2026/blob/main/ML/logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [1]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [2]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [3]:
# TODO: виведи баланс класів
# Частка кожного класу в цільовій змінній upgraded
print("Баланс класів:")
print(clients["upgraded"].value_counts(normalize=True))

# Визначаємо числові та категорійні стовпці
numeric_cols = clients.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = clients.select_dtypes(include=["object"]).columns.tolist()

print("\nЧислові ознаки:")
print(numeric_cols)

print("\nКатегорійні ознаки:")
print(categorical_cols)

Баланс класів:
upgraded
0    0.515556
1    0.484444
Name: proportion, dtype: float64

Числові ознаки:
['age', 'tenure', 'usage', 'support', 'upgraded']

Категорійні ознаки:
['plan', 'region']


✍️ Випиши списки стовпців (знадобляться далі):
> числові: ['age', 'tenure', 'usage', 'support']  
> категорійні: ['plan', 'region']  


### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [4]:
from sklearn.model_selection import train_test_split

# Відокремлюємо ознаки (X) і цільову змінну (y)
X = clients.drop("upgraded", axis=1)
y = clients["upgraded"]

# Поділ на train і test (20% тестових даних)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Розмір X_train:", X_train.shape)
print("Розмір X_test:", X_test.shape)

print("\nБаланс класів у train:")
print(y_train.value_counts(normalize=True))

print("\nБаланс класів у test:")
print(y_test.value_counts(normalize=True))

Розмір X_train: (720, 6)
Розмір X_test: (180, 6)

Баланс класів у train:
upgraded
0    0.515278
1    0.484722
Name: proportion, dtype: float64

Баланс класів у test:
upgraded
0    0.516667
1    0.483333
Name: proportion, dtype: float64


### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Списки ознак за типами даних
num_cols = ["age", "tenure", "usage", "support"]
cat_cols = ["plan", "region"]

# Створюємо препроцесор
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

print("Preprocess готовий")

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: задай num_cols, cat_cols і збери preprocess

# Списки ознак за типами даних
num_cols = ["age", "tenure", "usage", "support"]
cat_cols = ["plan", "region"]

# Створюємо препроцесор
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

print("Preprocess готовий")

Preprocess готовий


### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TODO: збери pipe

# Створюємо Pipeline: спочатку підготовка даних, потім модель
pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

print("Pipeline готовий")

Pipeline готовий


### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [8]:
# TODO: навчи pipe і виведи accuracy на тесті
# Навчання Pipeline на тренувальних даних
pipe.fit(X_train, y_train)

# Оцінка точності на тестових даних
accuracy = pipe.score(X_test, y_test)

print(f"Accuracy на тесті: {accuracy:.3f}")

Accuracy на тесті: 0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [9]:
from sklearn.metrics import confusion_matrix, classification_report

# TODO: передбач, виведи матрицю плутанини та звіт
y_pred = pipe.predict(X_test)

# Матриця плутанини
cm = confusion_matrix(y_test, y_pred)

print("Матриця плутанини:")
print(cm)

# Детальний звіт
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Матриця плутанини:
[[80 13]
 [15 72]]

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        93
           1       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [10]:
from sklearn.metrics import roc_auc_score

# TODO: дістань proba та порахуй ROC-AUC

# Ймовірність переходу на Premium (клас 1)
proba = pipe.predict_proba(X_test)[:, 1]

# Розрахунок ROC-AUC
auc = roc_auc_score(y_test, proba)

print(f"ROC-AUC: {auc:.3f}")

ROC-AUC: 0.927


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [11]:
# TODO: побудуй таблицю "ознака — коефіцієнт", відсортовану за |коеф.|

# Отримуємо назви ознак після One-Hot кодування
names = pipe.named_steps["prep"].get_feature_names_out()

# Отримуємо коефіцієнти логістичної регресії
coefs = pipe.named_steps["model"].coef_[0]

# Створюємо таблицю з коефіцієнтами
coef_df = pd.DataFrame({
    "feature": names,
    "coefficient": coefs
})

# Сортуємо за абсолютним значенням впливу
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

print(coef_df)

                feature  coefficient  abs_coef
2            num__usage     2.027135  2.027135
1           num__tenure     1.648386  1.648386
6    cat__plan_сімейний     1.411953  1.411953
4     cat__plan_базовий    -1.339728  1.339728
3          num__support    -0.835947  0.835947
7     cat__region_захід     0.537637  0.537637
0              num__age    -0.406875  0.406875
10     cat__region_схід    -0.340620  0.340620
8   cat__region_південь    -0.189350  0.189350
9    cat__region_північ     0.136239  0.136239
5    cat__plan_стандарт     0.071680  0.071680


✍️ **Відповідь словами:**
> Найсильніший позитивний фактор — `usage`, а найсильніший негативний — `support` (разом із базовим тарифом).  

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [13]:
# TODO: створи new_client, виведи рішення та ймовірність переходу
# Новий клієнт
new_client = pd.DataFrame([{
    "age": 30,
    "tenure": 24,
    "usage": 120,
    "support": 0,
    "plan": "сімейний",
    "region": "захід"
}])

# Ймовірність переходу на Premium
prob = pipe.predict_proba(new_client)[0, 1]

# Прогноз класу (0 або 1)
pred = pipe.predict(new_client)[0]

print(f"Ймовірність переходу на Premium: {prob:.3f}")
print(f"Рішення моделі: {pred} ({'перейде' if pred == 1 else 'не перейде'})")

Ймовірність переходу на Premium: 0.995
Рішення моделі: 1 (перейде)


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [14]:
from sklearn.model_selection import cross_val_score

# TODO: крос-валідація, виведи середнє ± розкид
import numpy as np

# Крос-валідація ROC-AUC
scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")

print("ROC-AUC по фолдах:", scores)
print(f"Середнє ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")

ROC-AUC по фолдах: [0.93437152 0.93140527 0.94005685 0.8907428  0.92070158]
Середнє ROC-AUC: 0.923 ± 0.018


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [ ]:
# Місце для бонусних експериментів

---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?
2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?
3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?
4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?
5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.

1. Логістична регресія — класифікація, бо прогнозує ймовірність класу (0/1), а не число.  
2. `predict_proba` показує ймовірність кожного класу, дозволяє оцінювати впевненість і змінювати поріг рішення.  
3. `Pipeline` захищає від data leakage: масштабування робиться тільки на `train`, а не з участю `test`.  
4. Логістика чутлива до масштабу (коефіцієнти + регуляризація), дерева — ні, бо працюють через пороги, а не відстані.  
5. `support` < 0 → більше звернень у підтримку = менша ймовірність переходу на преміум.  